# Runnable Router Reference

Developer-facing statements defined in `langchain_core.runnables.router`.

# `RouterInput: TypedDict`

`RouterInput` contains the routing key and the value passed to the selected `Runnable`.

## Fields

```python
key: str # Key used to select one Runnable
input: Any # Value passed to the selected Runnable
```


# `RouterRunnable: RunnableSerializable[RouterInput, Output]`

`RouterRunnable` selects one named `Runnable` using `input["key"]` and executes it with `input["input"]`.

## Type Parameter

```python
Output # Output type produced by the selected Runnable
```

## Field

```python
runnables: Mapping[str, Runnable[Any, Output]] # Route keys mapped to executable Runnables
```

## Constructor

```python
RouterRunnable(
    runnables: Mapping[
        str,
        Runnable[Any, Output] | Callable[[Any], Output],
    ], # Route keys mapped to Runnables or Python callables
) -> None # Initialize the router
```

Python callables are automatically converted into `Runnable` objects.

## Input and Output

```python
RouterInput # Dictionary containing the route key and selected Runnable input
Output # Result returned by the selected Runnable
```

## Overridden Properties and Methods

### `config_specs`

Returns the unique configurable-field specifications collected from every registered `Runnable`.

### `is_lc_serializable`

Returns `True`, indicating that `RouterRunnable` supports LangChain serialization.

### `get_lc_namespace`

Returns the LangChain Runnable serialization namespace.

### `invoke`

Synchronously selects the matching `Runnable` and invokes it with the value stored under `input`.

Raises `ValueError` when no Runnable is registered for the supplied key.

### `ainvoke`

Asynchronously selects the matching `Runnable` and invokes it with the value stored under `input`.

Raises `ValueError` when no Runnable is registered for the supplied key.

### `batch`

Routes each input independently and executes the selected Runnables concurrently.

Returns results in the same order as the input list.

When `return_exceptions=True`, execution exceptions are returned as result values.

Raises `ValueError` before execution when any routing key is unknown.

### `abatch`

Routes each input independently and asynchronously executes the selected Runnables.

The configured `max_concurrency` value limits concurrent asynchronous operations.

Returns results in the same order as the input list.

When `return_exceptions=True`, execution exceptions are returned as result values.

Raises `ValueError` before execution when any routing key is unknown.

### `stream`

Synchronously selects one Runnable and yields its streamed output.

Raises `ValueError` when the routing key is unknown.

### `astream`

Asynchronously selects one Runnable and yields its streamed output.

Raises `ValueError` when the routing key is unknown.

In [ ]:
from langchain_core.runnables.router import RouterInput, RouterRunnable # Import the router class and its input type

def square(number: int) -> int: # Define the operation used by the square route
    return number ** 2 # Return the square of the supplied number

def cube(number: int) -> int: # Define the operation used by the cube route
    return number ** 3 # Return the cube of the supplied number

router: RouterRunnable[int] = RouterRunnable( # Create the Runnable router
    runnables={ # Define the available route keys
        "square": square, # Map the square key to the square function
        "cube": cube, # Map the cube key to the cube function
    }, # Finish defining the routes
) # Finish creating RouterRunnable

square_input: RouterInput = { # Create input for the square route
    "key": "square", # Select the square Runnable
    "input": 4, # Pass 4 to the selected Runnable
} # Finish creating the square input

cube_input: RouterInput = { # Create input for the cube route
    "key": "cube", # Select the cube Runnable
    "input": 4, # Pass 4 to the selected Runnable
} # Finish creating the cube input

square_result: int = router.invoke(square_input) # Execute the square route

cube_result: int = router.invoke(cube_input) # Execute the cube route

batch_results: list[int] = router.batch( # Execute multiple routed inputs
    [square_input, cube_input] # Supply inputs that select different routes
) # Finish the batch execution

print("Square result:", square_result) # Display the square result

print("Cube result:", cube_result) # Display the cube result

print("Batch results:", batch_results) # Display the ordered batch results

## Routing Behaviour

- Routing uses the exact string stored in `input["key"]`.
- The selected Runnable receives only the value stored in `input["input"]`.
- Only one Runnable is selected for each input.
- Registered callables are converted into `RunnableLambda`-compatible Runnables.
- An empty batch returns an empty list.
- Batch inputs may select different Runnables.
- Batch execution uses the configuration associated with each input.
- Unknown routing keys raise `ValueError`.

## Developer-Facing Top-Level Statements

```python
RouterInput # Typed dictionary containing a route key and Runnable input
RouterRunnable # Serializable Runnable that selects a named Runnable
```